#  Credit Default Risk Prediction

## 📋 Table of Contents
1. [Setup & Data Loading](#1)
2. [Exploratory Data Analysis (EDA)](#2)
3. [Preprocessing & Feature Engineering](#3)
4. [Handling Class Imbalance](#4)
5. [Baseline Model — Logistic Regression](#5)
6. [Advanced Models — XGBoost & LightGBM](#6)
7. [Hyperparameter Tuning with Optuna](#7)
8. [Model Evaluation & Threshold Tuning](#8)
9. [Explainability with SHAP](#9)
10. [Scorecard & Final Submission](#10)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay
)
import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import shap

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('✅ All imports successful')

✅ All imports successful


## Step 0 — Download the Dataset

Before running this notebook, download the dataset from Kaggle:

```bash
# Option A: Kaggle CLI (recommended)
pip install kaggle
kaggle competitions download -c GiveMeSomeCredit
unzip GiveMeSomeCredit.zip -d data/

# Option B: Manual download
# Go to https://www.kaggle.com/c/GiveMeSomeCredit/data
# Download cs-training.csv and cs-test.csv into a ./data/ folder
```

Files you need:
- `data/cs-training.csv` — labelled training data (150,000 rows)
- `data/cs-test.csv` — unlabelled test data for Kaggle submission
- `data/sampleEntry.csv` — submission format reference

<a id='1'></a>
## 1. Setup & Data Loading

In [3]:
# Install any missing packages
!pip install xgboost lightgbm optuna imbalanced-learn shap --quiet

^C


In [ ]:
# Load data
train_raw = pd.read_csv('data/cs-training.csv', index_col=0)
test_raw  = pd.read_csv('data/cs-test.csv',  index_col=0)

print(f'Training shape : {train_raw.shape}')
print(f'Test shape     : {test_raw.shape}')
train_raw.head()

In [ ]:
# Quick data types & non-null counts
train_raw.info()

In [ ]:
# Summary statistics
train_raw.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

<a id='2'></a>
## 2. Exploratory Data Analysis (EDA)

In [ ]:
cc

In [ ]:
# ── 2.2 Missing values ────────────────────────────────────────────────────────
missing = (train_raw.isnull().sum() / len(train_raw) * 100).sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(8, 3))
missing.plot(kind='barh', color='#E8604C', ax=ax)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Missing Values by Feature', fontweight='bold')
ax.set_xlabel('% missing')
for i, v in enumerate(missing.values):
    ax.text(v + 0.2, i, f'{v:.1f}%', va='center')
plt.tight_layout()
plt.show()

print('\nMissing value counts:')
print(train_raw.isnull().sum()[train_raw.isnull().sum() > 0])

In [ ]:
# ── 2.3 Feature distributions ─────────────────────────────────────────────────
FEATURES = [c for c in train_raw.columns if c != TARGET]

fig, axes = plt.subplots(3, 4, figsize=(18, 11))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    ax = axes[i]
    for label, color in [(0, '#4C9BE8'), (1, '#E8604C')]:
        subset = train_raw[train_raw[TARGET] == label][col].dropna()
        # Cap extreme outliers for visualisation only
        cap = subset.quantile(0.99)
        subset = subset.clip(upper=cap)
        ax.hist(subset, bins=40, alpha=0.55, color=color,
                label='Default' if label == 1 else 'No Default', density=True)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=7)

# Hide extra subplot
for j in range(len(FEATURES), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Default vs Non-Default (99th pct capped)', 
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4 Correlation heatmap ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 9))
corr = train_raw.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.5 Outlier inspection ────────────────────────────────────────────────────
# Age = 0 is clearly invalid
print('Age == 0 records  :', (train_raw['age'] == 0).sum())

# NumberOfDependents extremes
print('Max dependents    :', train_raw['NumberOfDependents'].max())

# RevolvingUtilization > 1 (>100% utilization)
print('Utilization > 1   :', (train_raw['RevolvingUtilizationOfUnsecuredLines'] > 1).sum())

# DebtRatio extremes
print('DebtRatio > 1     :', (train_raw['DebtRatio'] > 1).sum())

<a id='3'></a>
## 3. Preprocessing & Feature Engineering

In [ ]:
def preprocess(df, is_train=True):
    """
    Full preprocessing + feature engineering pipeline.
    Returns X (features), y (target if train), and feature names.
    """
    df = df.copy()

    # ── 3.1 Fix known data quality issues ─────────────────────────────────────
    # Age = 0 → replace with median
    df.loc[df['age'] == 0, 'age'] = df.loc[df['age'] > 0, 'age'].median()

    # Cap RevolvingUtilization at 1 (anything > 1 is erroneous or maxed-out)
    df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(0, 1)

    # Cap DebtRatio at 99th percentile to suppress outliers
    debt_cap = df['DebtRatio'].quantile(0.99)
    df['DebtRatio'] = df['DebtRatio'].clip(0, debt_cap)

    # ── 3.2 Impute missing values ──────────────────────────────────────────────
    # MonthlyIncome: median imputation
    df['MonthlyIncome'].fillna(df['MonthlyIncome'].median(), inplace=True)

    # NumberOfDependents: mode imputation (usually 0)
    df['NumberOfDependents'].fillna(df['NumberOfDependents'].mode()[0], inplace=True)

    # ── 3.3 Feature engineering ────────────────────────────────────────────────
    # Delinquency aggregates
    df['TotalLate'] = (
        df['NumberOfTime30-59DaysPastDueNotWorse'] +
        df['NumberOfTime60-89DaysPastDueNotWorse'] +
        df['NumberOfTimes90DaysLate']
    )
    df['WeightedLate'] = (
        df['NumberOfTime30-59DaysPastDueNotWorse'] * 1 +
        df['NumberOfTime60-89DaysPastDueNotWorse'] * 2 +
        df['NumberOfTimes90DaysLate'] * 3
    )

    # Income per dependent (add 1 to avoid division by zero)
    df['IncomePerDependent'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1)

    # Estimated monthly debt payment
    df['MonthlyDebtPayment'] = df['DebtRatio'] * df['MonthlyIncome']

    # Indicator: Has any open credit lines
    df['HasOpenCredit'] = (df['NumberOfOpenCreditLinesAndLoans'] > 0).astype(int)

    # Indicator: Has real estate loans
    df['HasRealEstate'] = (df['NumberRealEstateLoansOrLines'] > 0).astype(int)

    # Age bands
    df['AgeBand'] = pd.cut(df['age'],
                           bins=[0, 25, 35, 45, 55, 65, 120],
                           labels=[0, 1, 2, 3, 4, 5]).astype(int)

    # Log transforms for heavily skewed columns
    for col in ['MonthlyIncome', 'DebtRatio', 'IncomePerDependent', 'MonthlyDebtPayment']:
        df[f'log_{col}'] = np.log1p(df[col])

    # Binary flag: ever seriously delinquent
    df['EverSerious'] = (df['NumberOfTimes90DaysLate'] > 0).astype(int)

    # ── 3.4 Separate target ────────────────────────────────────────────────────
    if is_train:
        y = df[TARGET]
        X = df.drop(columns=[TARGET])
        return X, y
    else:
        return df, None


X, y = preprocess(train_raw, is_train=True)
X_test_final, _ = preprocess(test_raw, is_train=False)

print(f'Feature matrix shape : {X.shape}')
print(f'Target shape         : {y.shape}')
print(f'\nEngineered features  : {list(X.columns)}')

In [ ]:
# Train / validation split (stratified to preserve class balance)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f'Train size : {X_train.shape[0]:,}  |  default rate: {y_train.mean():.3f}')
print(f'Val size   : {X_val.shape[0]:,}  |  default rate: {y_val.mean():.3f}')

<a id='4'></a>
## 4. Handling Class Imbalance

We compare three strategies:
| Strategy | How it works |
|---|---|
| `class_weight='balanced'` | Upweights minority class in loss function |
| SMOTE | Synthesises new minority samples in feature space |
| Threshold tuning | Use default weights, shift decision threshold post-training |

In [ ]:
# ── Strategy 1: class_weight='balanced' (built into sklearn estimators) ────────
# Used directly in model constructors below — no transformation needed here.

# ── Strategy 2: SMOTE oversampling ────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('Before SMOTE :', pd.Series(y_train).value_counts().to_dict())
print('After  SMOTE :', pd.Series(y_train_smote).value_counts().to_dict())

<a id='5'></a>
## 5. Baseline Model — Logistic Regression

A regularised logistic regression is our interpretable baseline. In production credit scoring, regulators often require explainability — logistic regression with Weight-of-Evidence features is still widely deployed.

In [ ]:
# Scale + fit
scaler = RobustScaler()   # robust to outliers
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

lr = LogisticRegression(
    C=0.1,
    class_weight='balanced',
    max_iter=1000,
    random_state=SEED
)
lr.fit(X_train_sc, y_train)

lr_prob = lr.predict_proba(X_val_sc)[:, 1]

print(f'Logistic Regression')
print(f'  AUC-ROC   : {roc_auc_score(y_val, lr_prob):.4f}')
print(f'  PR-AUC    : {average_precision_score(y_val, lr_prob):.4f}')

In [ ]:
# Feature coefficients (interpretability)
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#E8604C' if c > 0 else '#4C9BE8' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression — Top 15 Feature Coefficients', fontweight='bold')
ax.set_xlabel('Coefficient (positive = higher default risk)')
plt.tight_layout()
plt.show()

NameError: name 'pd' is not defined

<a id='6'></a>
## 6. Advanced Models — XGBoost & LightGBM

In [ ]:
# ── 6.1 XGBoost ───────────────────────────────────────────────────────────────
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight = {scale_pos:.2f}  (ratio of neg : pos for XGBoost)')

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=SEED,
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_prob = xgb_model.predict_proba(X_val)[:, 1]
print(f'\nXGBoost')
print(f'  AUC-ROC   : {roc_auc_score(y_val, xgb_prob):.4f}')
print(f'  PR-AUC    : {average_precision_score(y_val, xgb_prob):.4f}')

In [ ]:
# ── 6.2 LightGBM ──────────────────────────────────────────────────────────────
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(-1)]
)

lgb_prob = lgb_model.predict_proba(X_val)[:, 1]
print(f'LightGBM')
print(f'  AUC-ROC   : {roc_auc_score(y_val, lgb_prob):.4f}')
print(f'  PR-AUC    : {average_precision_score(y_val, lgb_prob):.4f}')

In [ ]:
# ── 6.3 Compare ROC curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

models_info = [
    ('Logistic Regression', lr_prob,  '#888888'),
    ('XGBoost',             xgb_prob, '#E8604C'),
    ('LightGBM',            lgb_prob, '#4C9BE8'),
]

# ROC
ax = axes[0]
for name, prob, color in models_info:
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc = roc_auc_score(y_val, prob)
    ax.plot(fpr, tpr, label=f'{name}  (AUC={auc:.4f})', color=color, lw=2)
ax.plot([0,1],[0,1], 'k--', lw=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(fontsize=9)

# Precision-Recall
ax = axes[1]
for name, prob, color in models_info:
    prec, rec, _ = precision_recall_curve(y_val, prob)
    pr_auc = average_precision_score(y_val, prob)
    ax.plot(rec, prec, label=f'{name}  (PR-AUC={pr_auc:.4f})', color=color, lw=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

<a id='7'></a>
## 7. Hyperparameter Tuning with Optuna

We tune LightGBM (usually fastest). Optuna uses Tree-structured Parzen Estimators (TPE) — a Bayesian optimisation approach more efficient than grid search.

In [ ]:
def objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 200, 1000),
        'max_depth'        : trial.suggest_int('max_depth', 3, 10),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 150),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'is_unbalance'     : True,
        'random_state'     : SEED,
        'n_jobs'           : -1,
        'verbose'          : -1,
    }
    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X_train, y_train, cv=cv,
                             scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nBest AUC-ROC (CV) : {study.best_value:.4f}')
print(f'Best params       : {study.best_params}')

In [ ]:
# Retrain best model on full training set
best_params = study.best_params
best_params.update({'is_unbalance': True, 'random_state': SEED, 'n_jobs': -1, 'verbose': -1})

best_lgb = lgb.LGBMClassifier(**best_params)
best_lgb.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             callbacks=[lgb.early_stopping(50, verbose=False),
                        lgb.log_evaluation(-1)])

best_prob = best_lgb.predict_proba(X_val)[:, 1]
print(f'\nTuned LightGBM')
print(f'  AUC-ROC   : {roc_auc_score(y_val, best_prob):.4f}')
print(f'  PR-AUC    : {average_precision_score(y_val, best_prob):.4f}')

<a id='8'></a>
## 8. Model Evaluation & Threshold Tuning

The default 0.5 threshold is rarely optimal for imbalanced problems. We find the threshold that maximises the **F1 score** (balances precision & recall). In practice you may also use a business-driven threshold (e.g. maximum recall at 80% precision).

In [ ]:
# ── 8.1 Threshold search ──────────────────────────────────────────────────────
from sklearn.metrics import f1_score

thresholds = np.linspace(0.01, 0.99, 200)
f1_scores  = [f1_score(y_val, (best_prob >= t).astype(int)) for t in thresholds]

best_threshold = thresholds[np.argmax(f1_scores)]
print(f'Optimal threshold (max F1) : {best_threshold:.3f}')
print(f'Max F1 score               : {max(f1_scores):.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, f1_scores, color='#4C9BE8', lw=2)
ax.axvline(best_threshold, color='#E8604C', linestyle='--', lw=1.5,
           label=f'Optimal threshold = {best_threshold:.3f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('F1 Score')
ax.set_title('Threshold vs F1 Score', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.2 Confusion matrix at optimal threshold ─────────────────────────────────
y_pred_opt = (best_prob >= best_threshold).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Default threshold (0.5)
y_pred_default = (best_prob >= 0.5).astype(int)
ConfusionMatrixDisplay(confusion_matrix(y_val, y_pred_default),
                       display_labels=['No Default', 'Default']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Threshold = 0.50', fontweight='bold')

# Optimal threshold
ConfusionMatrixDisplay(confusion_matrix(y_val, y_pred_opt),
                       display_labels=['No Default', 'Default']).plot(ax=axes[1], colorbar=False)
axes[1].set_title(f'Threshold = {best_threshold:.3f} (optimal F1)', fontweight='bold')

plt.suptitle('Confusion Matrices: Default vs Optimal Threshold', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print('\n── Classification Report (optimal threshold) ─────────────────────────────')
print(classification_report(y_val, y_pred_opt, target_names=['No Default', 'Default']))

In [ ]:
# ── 8.3 Gini coefficient & KS statistic ──────────────────────────────────────
auc = roc_auc_score(y_val, best_prob)
gini = 2 * auc - 1

# KS stat = max separation between TPR and FPR
fpr, tpr, _ = roc_curve(y_val, best_prob)
ks_stat = np.max(np.abs(tpr - fpr))

print(f'AUC-ROC  : {auc:.4f}')
print(f'Gini     : {gini:.4f}   (>0.4 is good in credit scoring)')
print(f'KS stat  : {ks_stat:.4f}   (>0.4 is good in credit scoring)')

<a id='9'></a>
## 9. Explainability with SHAP

SHAP (SHapley Additive exPlanations) provides:
- **Global importance** — which features drive the model most overall  
- **Individual explanations** — why a specific customer was predicted high/low risk

In [ ]:
# ── 9.1 Compute SHAP values ───────────────────────────────────────────────────
explainer   = shap.TreeExplainer(best_lgb)
shap_values = explainer.shap_values(X_val)

# For binary classification, shap_values is a list [class0, class1] in some versions
if isinstance(shap_values, list):
    sv = shap_values[1]   # values for class 1 (Default)
else:
    sv = shap_values

print(f'SHAP values shape: {sv.shape}')

In [ ]:
# ── 9.2 Global feature importance (beeswarm plot) ─────────────────────────────
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_val, plot_type='dot', max_display=15,
                  show=False)
plt.title('SHAP Summary Plot — Feature Impact on Default Probability', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.3 Bar chart of mean |SHAP| ─────────────────────────────────────────────
plt.figure(figsize=(9, 6))
shap.summary_plot(sv, X_val, plot_type='bar', max_display=15, show=False)
plt.title('Mean Absolute SHAP Values (Global Importance)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.4 Individual explanation — highest risk customer ───────────────────────
high_risk_idx = np.argmax(best_prob)
print(f'Highest risk customer (val index {high_risk_idx})')
print(f'Predicted probability : {best_prob[high_risk_idx]:.4f}')
print(f'Actual label          : {y_val.iloc[high_risk_idx]}')

shap.waterfall_plot(
    shap.Explanation(
        values=sv[high_risk_idx],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, list)
                    else explainer.expected_value[1],
        data=X_val.iloc[high_risk_idx].values,
        feature_names=X_val.columns.tolist()
    ),
    max_display=12,
    show=True
)

In [ ]:
# ── 9.5 SHAP dependence plot — top feature ────────────────────────────────────
top_feature = X_val.columns[np.abs(sv).mean(axis=0).argmax()]
print(f'Top SHAP feature: {top_feature}')

plt.figure(figsize=(8, 5))
shap.dependence_plot(top_feature, sv, X_val,
                     interaction_index='auto', show=False)
plt.title(f'SHAP Dependence — {top_feature}', fontweight='bold')
plt.tight_layout()
plt.show()

<a id='10'></a>
## 10. Scorecard & Final Submission

In [ ]:
# ── 10.1 Final model summary ──────────────────────────────────────────────────
results = {
    'Model'   : ['Logistic Regression', 'XGBoost', 'LightGBM (default)', 'LightGBM (tuned)'],
    'AUC-ROC' : [
        roc_auc_score(y_val, lr_prob),
        roc_auc_score(y_val, xgb_prob),
        roc_auc_score(y_val, lgb_prob),
        roc_auc_score(y_val, best_prob)
    ],
    'PR-AUC'  : [
        average_precision_score(y_val, lr_prob),
        average_precision_score(y_val, xgb_prob),
        average_precision_score(y_val, lgb_prob),
        average_precision_score(y_val, best_prob)
    ]
}

results_df = pd.DataFrame(results)
results_df.style \
    .highlight_max(subset=['AUC-ROC', 'PR-AUC'], color='#c8f0d0') \
    .format({'AUC-ROC': '{:.4f}', 'PR-AUC': '{:.4f}'})

In [ ]:
# ── 10.2 Generate Kaggle submission file ──────────────────────────────────────
test_probs = best_lgb.predict_proba(X_test_final)[:, 1]

submission = pd.DataFrame({
    'Id'      : test_raw.index,
    'Probability': test_probs
})

submission.to_csv('submission.csv', index=False)
print('✅ submission.csv saved')
submission.head(10)

In [ ]:
# ── 10.3 Score distribution on test set ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Validation set — coloured by true label
axes[0].hist(best_prob[y_val == 0], bins=60, alpha=0.6, color='#4C9BE8', label='No Default', density=True)
axes[0].hist(best_prob[y_val == 1], bins=60, alpha=0.6, color='#E8604C', label='Default', density=True)
axes[0].axvline(best_threshold, color='black', linestyle='--', lw=1.5, label=f'Threshold={best_threshold:.3f}')
axes[0].set_title('Validation: Score Distribution by Label', fontweight='bold')
axes[0].set_xlabel('Predicted Default Probability')
axes[0].legend()

# Test set
axes[1].hist(test_probs, bins=60, color='#7F77DD', alpha=0.8)
axes[1].axvline(best_threshold, color='#E8604C', linestyle='--', lw=1.5, label=f'Threshold={best_threshold:.3f}')
axes[1].set_title('Test Set: Predicted Score Distribution', fontweight='bold')
axes[1].set_xlabel('Predicted Default Probability')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nTest set default rate (at threshold) : {(test_probs >= best_threshold).mean():.3f}')

## ✅ Project Complete!

### What you built:
| Step | What was done |
|---|---|
| EDA | Class imbalance, missing values, distributions, outliers |
| Preprocessing | Age fix, utilization cap, debt ratio clip, median imputation |
| Feature Engineering | Delinquency aggregates, income ratios, log transforms, age bands |
| Imbalance | SMOTE + `scale_pos_weight` / `is_unbalance` strategies |
| Modeling | Logistic Regression → XGBoost → LightGBM |
| Tuning | Optuna (50 TPE trials, 3-fold CV) |
| Evaluation | AUC-ROC, PR-AUC, Gini, KS, optimal threshold |
| Explainability | SHAP beeswarm, bar, waterfall, dependence plots |
| Submission | Kaggle-ready `submission.csv` |

### 🚀 Ways to push further:
- **Stacking ensemble**: combine LightGBM + XGBoost + Logistic Regression predictions as meta-features
- **Weight-of-Evidence encoding**: bin continuous features and encode using WoE for a proper credit scorecard
- **Calibration**: use `CalibratedClassifierCV` if you need well-calibrated probabilities
- **Monitoring**: track PSI (Population Stability Index) to detect dataset drift in production